In [1]:
import speech_recognition as sr
import pyttsx3
import time

class SpeechRecognizer:
    def __init__(self):
        # Initialize the recognizer
        self.recognizer = sr.Recognizer()
        self.recognizer.energy_threshold = 300  # minimum audio energy to consider for recording
        self.recognizer.dynamic_energy_threshold = True
        self.recognizer.pause_threshold = 0.8  # seconds of non-speaking audio before a phrase is considered complete
        
        # Initialize text-to-speech engine
        self.engine = pyttsx3.init()
        voices = self.engine.getProperty('voices')
        self.engine.setProperty('voice', voices[0].id)  # 0 for male, 1 for female
        
        # Microphone setup
        self.microphone = sr.Microphone()
        with self.microphone as source:
            self.recognizer.adjust_for_ambient_noise(source)  # calibrate for ambient noise
        
    def listen(self):
        """Listen to microphone input and return recognized text"""
        with self.microphone as source:
            print("Listening...")
            audio = self.recognizer.listen(source)
            
        try:
            text = self.recognizer.recognize_google(audio)
            print(f"You said: {text}")
            return text
        except sr.UnknownValueError:
            print("Google Speech Recognition could not understand audio")
            return None
        except sr.RequestError as e:
            print(f"Could not request results from Google Speech Recognition service; {e}")
            return None
            
    def speak(self, text):
        """Convert text to speech"""
        print(f"Speaking: {text}")
        self.engine.say(text)
        self.engine.runAndWait()
        
    def run_interactive_mode(self):
        """Run in interactive mode where it listens and responds"""
        self.speak("Hello! I'm your speech recognition assistant. How can I help you today?")
        
        while True:
            try:
                text = self.listen()
                if text:
                    if "exit" in text.lower() or "quit" in text.lower():
                        self.speak("Goodbye!")
                        break
                    elif "hello" in text.lower():
                        self.speak("Hello there!")
                    elif "time" in text.lower():
                        current_time = time.strftime("%I:%M %p")
                        self.speak(f"The current time is {current_time}")
                    elif "date" in text.lower():
                        current_date = time.strftime("%A, %B %d, %Y")
                        self.speak(f"Today is {current_date}")
                    else:
                        self.speak(f"You said: {text}")
            except KeyboardInterrupt:
                self.speak("Goodbye!")
                break

if __name__ == "__main__":
    recognizer = SpeechRecognizer()
    recognizer.run_interactive_mode()

Speaking: Hello! I'm your speech recognition assistant. How can I help you today?
Listening...
You said: what is date today
Speaking: Today is Saturday, May 03, 2025
Listening...
You said: what is time today
Speaking: The current time is 10:43 AM
Listening...
You said: what is date today date
Speaking: Today is Saturday, May 03, 2025
Listening...
You said: exit
Speaking: Goodbye!


In [7]:
import os
import time
import wave
import boto3
from botocore.exceptions import BotoCoreError, ClientError
import speech_recognition as sr
from pydub import AudioSegment
import io

class SpeechRecognizer:
    def __init__(self):
        # Initialize local recognizer
        self.local_recognizer = sr.Recognizer()
        self.local_recognizer.energy_threshold = 400
        self.local_recognizer.dynamic_energy_threshold = True
        self.local_recognizer.pause_threshold = 1.0
        
        # Initialize AWS clients
        self.aws_transcribe = boto3.client('transcribe')
        self.aws_s3 = boto3.client('s3')
        self.aws_polly = boto3.client('polly')
        
        # Configuration
        self.bucket_name = "your-audio-bucket-name"  # Change this to your S3 bucket
        self.region_name = "us-east-1"  # Change to your region
        
        # Microphone setup
        self.microphone = sr.Microphone()
        with self.microphone as source:
            self.local_recognizer.adjust_for_ambient_noise(source)
    
    def record_audio(self, duration=5, filename="recording.wav"):
        """Record audio from microphone and save to file"""
        with self.microphone as source:
            print(f"Recording for {duration} seconds...")
            audio = self.local_recognizer.listen(source, timeout=None, phrase_time_limit=duration)
        
        with open(filename, "wb") as f:
            f.write(audio.get_wav_data())
        return filename
    
    def convert_to_required_format(self, input_file, output_file="converted.wav"):
        """Convert audio to format suitable for AWS Transcribe"""
        audio = AudioSegment.from_file(input_file)
        audio = audio.set_channels(1)  # Mono
        audio = audio.set_frame_rate(16000)  # 16kHz
        audio.export(output_file, format="wav")
        return output_file
    
    def upload_to_s3(self, file_path):
        """Upload audio file to S3 bucket"""
        try:
            key = os.path.basename(file_path)
            self.aws_s3.upload_file(file_path, self.bucket_name, key)
            s3_uri = f"s3://{self.bucket_name}/{key}"
            return s3_uri
        except (BotoCoreError, ClientError) as error:
            print(f"Error uploading to S3: {error}")
            return None
    
    def transcribe_with_aws(self, s3_uri, language_code='en-US'):
        """Transcribe audio using AWS Transcribe"""
        job_name = f"transcribe_job_{int(time.time())}"
        
        try:
            self.aws_transcribe.start_transcription_job(
                TranscriptionJobName=job_name,
                Media={'MediaFileUri': s3_uri},
                MediaFormat='wav',
                LanguageCode=language_code,
                OutputBucketName=self.bucket_name
            )
            
            while True:
                status = self.aws_transcribe.get_transcription_job(TranscriptionJobName=job_name)
                if status['TranscriptionJob']['TranscriptionJobStatus'] in ['COMPLETED', 'FAILED']:
                    break
                time.sleep(5)
            
            if status['TranscriptionJob']['TranscriptionJobStatus'] == 'COMPLETED':
                transcript_uri = status['TranscriptionJob']['Transcript']['TranscriptFileUri']
                # In a real app, you would download and parse the JSON transcript
                return f"Transcription complete. Results at: {transcript_uri}"
            else:
                return "Transcription job failed"
        except (BotoCoreError, ClientError) as error:
            return f"Error with AWS Transcribe: {error}"
    
    def transcribe_locally(self, audio_file):
        """Transcribe audio using local speech recognition"""
        with sr.AudioFile(audio_file) as source:
            audio = self.local_recognizer.record(source)
        
        try:
            # Using Google Web Speech API
            text = self.local_recognizer.recognize_google(audio)
            return text
        except sr.UnknownValueError:
            return "Could not understand audio"
        except sr.RequestError as e:
            return f"Could not request results; {e}"
    
    def synthesize_speech(self, text, output_file="output.mp3"):
        """Convert text to speech using AWS Polly"""
        try:
            response = self.aws_polly.synthesize_speech(
                Text=text,
                OutputFormat="mp3",
                VoiceId="Joanna"  # Change to your preferred voice
            )
            
            with open(output_file, "wb") as f:
                f.write(response['AudioStream'].read())
            return output_file
        except (BotoCoreError, ClientError) as error:
            print(f"Error with AWS Polly: {error}")
            return None
    
    def interactive_mode(self):
        """Run in interactive mode"""
        print("Speech Recognition System")
        print("1. Local transcription")
        print("2. AWS Transcribe")
        print("3. Text-to-Speech")
        print("4. Exit")
        
        while True:
            choice = input("Enter your choice (1-4): ")
            
            if choice == "1":
                audio_file = self.record_audio(duration=5)
                result = self.transcribe_locally(audio_file)
                print(f"Local Transcription: {result}")
                
            elif choice == "2":
                audio_file = self.record_audio(duration=10)
                converted_file = self.convert_to_required_format(audio_file)
                s3_uri = self.upload_to_s3(converted_file)
                if s3_uri:
                    result = self.transcribe_with_aws(s3_uri)
                    print(f"AWS Transcription: {result}")
                
            elif choice == "3":
                text = input("Enter text to convert to speech: ")
                output_file = self.synthesize_speech(text)
                if output_file:
                    print(f"Audio saved to {output_file}")
                
            elif choice == "4":
                print("Exiting...")
                break
                
            else:
                print("Invalid choice")

if __name__ == "__main__":
    recognizer = SpeechRecognizer()
    recognizer.interactive_mode()

F:\anacondaa\Lib\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


NoRegionError: You must specify a region.

In [5]:
!pip install pydub

In [13]:
import speech_recognition as sr
import pyttsx3
import os
import time
import tkinter as tk
from tkinter import ttk, filedialog
from vosk import Model, KaldiRecognizer
import json
import wave
import pyaudio
import boto3
from botocore.exceptions import BotoCoreError, ClientError

class SpeechRecognitionApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Speech Recognition App")
        self.root.geometry("600x400")

        # Initialize recognizers
        self.recognizer = sr.Recognizer()
        self.recognizer.energy_threshold = 300
        self.microphone = sr.Microphone()

        # Initialize TTS engine
        self.engine = pyttsx3.init()
        self.engine.setProperty('rate', 150)

        # Load Vosk model (offline)
        self.vosk_model = Model("models/vosk-model-small-en-us-0.15")
        self.vosk_recognizer = KaldiRecognizer(self.vosk_model, 16000)

        # AWS Transcribe setup (optional)
        self.aws_transcribe = boto3.client('transcribe', region_name='us-east-1')

        # GUI components
        self.setup_gui()

    def setup_gui(self):
        # Mode selection
        self.mode_var = tk.StringVar(value="google")
        ttk.Label(self.root, text="Recognition Mode:").pack(pady=5)
        ttk.Radiobutton(self.root, text="Google (Online)", variable=self.mode_var, value="google").pack()
        ttk.Radiobutton(self.root, text="Vosk (Offline)", variable=self.mode_var, value="vosk").pack()
        ttk.Radiobutton(self.root, text="AWS Transcribe (Cloud)", variable=self.mode_var, value="aws").pack()

        # Text output
        self.output_text = tk.Text(self.root, height=10, width=70)
        self.output_text.pack(pady=10)

        # Buttons
        ttk.Button(self.root, text="Start Listening", command=self.start_listening).pack(pady=5)
        ttk.Button(self.root, text="Stop Listening", command=self.stop_listening).pack(pady=5)
        ttk.Button(self.root, text="Speak Response", command=self.speak_response).pack(pady=5)
        ttk.Button(self.root, text="Clear Text", command=self.clear_text).pack(pady=5)

    def start_listening(self):
        self.output_text.insert(tk.END, "Listening... Speak now!\n")
        self.root.update()

        with self.microphone as source:
            audio = self.recognizer.listen(source, timeout=5)

        mode = self.mode_var.get()
        if mode == "google":
            self.process_with_google(audio)
        elif mode == "vosk":
            self.process_with_vosk(audio)
        elif mode == "aws":
            self.process_with_aws(audio)

    def process_with_google(self, audio):
        try:
            text = self.recognizer.recognize_google(audio)
            self.output_text.insert(tk.END, f"Google: {text}\n")
        except sr.UnknownValueError:
            self.output_text.insert(tk.END, "Google could not understand audio.\n")
        except sr.RequestError:
            self.output_text.insert(tk.END, "Google API unavailable.\n")

    def process_with_vosk(self, audio):
        wav_data = audio.get_wav_data()
        wf = wave.open("temp.wav", "wb")
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(16000)
        wf.writeframes(wav_data)
        wf.close()

        wf = wave.open("temp.wav", "rb")
        data = wf.readframes(4000)
        while data:
            if self.vosk_recognizer.AcceptWaveform(data):
                result = json.loads(self.vosk_recognizer.Result())
                if 'text' in result:
                    self.output_text.insert(tk.END, f"Vosk: {result['text']}\n")
            data = wf.readframes(4000)
        wf.close()

    def process_with_aws(self, audio):
        wav_data = audio.get_wav_data()
        with open("temp.wav", "wb") as f:
            f.write(wav_data)

        job_name = f"transcribe_job_{int(time.time())}"
        try:
            self.aws_transcribe.start_transcription_job(
                TranscriptionJobName=job_name,
                Media={'MediaFileUri': "temp.wav"},
                MediaFormat="wav",
                LanguageCode="en-US"
            )
            self.output_text.insert(tk.END, "AWS Transcribe job started. Check AWS Console for results.\n")
        except (BotoCoreError, ClientError) as e:
            self.output_text.insert(tk.END, f"AWS Error: {e}\n")

    def speak_response(self):
        text = self.output_text.get("1.0", tk.END).strip()
        if text:
            self.engine.say(text)
            self.engine.runAndWait()

    def stop_listening(self):
        self.output_text.insert(tk.END, "Stopped listening.\n")

    def clear_text(self):
        self.output_text.delete("1.0", tk.END)

if __name__ == "__main__":
    root = tk.Tk()
    app = SpeechRecognitionApp(root)
    root.mainloop()

Exception: Failed to create a model

In [11]:
!pip install vosk

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ---------------------------------------- 0.0/14.0 MB ? eta -:--:--
    --------------------------------------- 0.3/14.0 MB ? eta -:--:--
   -- ------------------------------------- 0.8/14.0 MB 2.4 MB/s eta 0:00:06
   --- ------------------------------------ 1.3/14.0 MB 2.3 MB/s eta 0:00:06
   ----- ---------------------------------- 1.8/14.0 MB 2.4 MB/s eta 0:00:06
   ------- -------------------------------- 2.6/14.0 MB 2.8 MB/s eta 0:00:05
   --------- ------------------------------ 3.4/14.0 MB 3.0 MB/s eta 0:00:04
   ------------- -------------------------- 4.7/14.0 MB 3.4 MB/s eta 0:00:03
   ---------------- ----------------------- 5.8/14.0 MB 3.7 MB/s eta 0:00:03
   -------------------- ------------------- 7.1/14.0 MB 3.9 MB/s eta 0:00:02
   ----------------------- ---------------- 8.4/14.0 MB 4.2 MB/s eta 0:00:02
   --------------------------- ------------ 9.7/14.0 MB 4.4 MB/s 

In [15]:
import speech_recognition as sr
import pyttsx3
import time
import os
from datetime import datetime

class SpeechRecognitionApp:
    def __init__(self):
        # Initialize speech recognizer and text-to-speech engine
        self.recognizer = sr.Recognizer()
        self.engine = pyttsx3.init()
        
        # Configure engine properties
        self.engine.setProperty('rate', 150)  # Speed of speech
        voices = self.engine.getProperty('voices')
        self.engine.setProperty('voice', voices[1].id)  # Change index for different voices
        
        # Set energy threshold for ambient noise
        self.recognizer.energy_threshold = 4000
        
    def speak(self, text):
        """Convert text to speech"""
        print(f"System: {text}")
        self.engine.say(text)
        self.engine.runAndWait()
        
    def listen(self):
        """Listen to microphone input and return recognized text"""
        with sr.Microphone() as source:
            print("Listening...")
            self.recognizer.adjust_for_ambient_noise(source)
            audio = self.recognizer.listen(source)
            
        try:
            print("Recognizing...")
            text = self.recognizer.recognize_google(audio)
            print(f"You said: {text}")
            return text.lower()
        except sr.UnknownValueError:
            self.speak("Sorry, I didn't catch that. Could you repeat?")
            return None
        except sr.RequestError as e:
            self.speak(f"Could not request results from Google Speech Recognition service; {e}")
            return None
            
    def recognize_audio_file(self, filename):
        """Recognize speech from an audio file"""
        try:
            with sr.AudioFile(filename) as source:
                audio = self.recognizer.record(source)
                text = self.recognizer.recognize_google(audio)
                print(f"Text from audio file: {text}")
                return text.lower()
        except FileNotFoundError:
            print(f"Error: File {filename} not found.")
            return None
        except sr.UnknownValueError:
            print("Google Speech Recognition could not understand audio")
            return None
        except sr.RequestError as e:
            print(f"Could not request results from Google Speech Recognition service; {e}")
            return None
            
    def process_command(self, command):
        """Process voice commands"""
        if command is None:
            return
            
        # Basic commands
        if "hello" in command:
            self.speak("Hello there! How can I help you?")
        elif "time" in command:
            current_time = datetime.now().strftime("%H:%M")
            self.speak(f"The current time is {current_time}")
        elif "date" in command:
            current_date = datetime.now().strftime("%B %d, %Y")
            self.speak(f"Today's date is {current_date}")
        elif "thank you" in command:
            self.speak("You're welcome!")
        elif "exit" in command or "quit" in command:
            self.speak("Goodbye!")
            return True
        else:
            self.speak("I didn't understand that command. Try saying hello, time, date, or exit.")
            
        return False
            
    def run_live_mode(self):
        """Run speech recognition in live microphone mode"""
        self.speak("Speech recognition system activated. How can I help you?")
        
        while True:
            command = self.listen()
            if self.process_command(command):
                break
                
    def run_file_mode(self, filename):
        """Run speech recognition on an audio file"""
        if not os.path.exists(filename):
            print(f"Error: File {filename} does not exist.")
            return
            
        self.speak(f"Processing audio file: {filename}")
        command = self.recognize_audio_file(filename)
        self.process_command(command)
        
    def test_microphone(self):
        """Test if microphone is working"""
        try:
            with sr.Microphone() as source:
                self.recognizer.adjust_for_ambient_noise(source)
                self.speak("Microphone test. Please say something...")
                audio = self.recognizer.listen(source, timeout=5)
                self.speak("Microphone is working properly!")
                return True
        except Exception as e:
            self.speak(f"Microphone test failed: {e}")
            return False

def main():
    app = SpeechRecognitionApp()
    
    print("""
    Speech Recognition Project
    -------------------------
    1. Live microphone mode
    2. Audio file mode
    3. Test microphone
    4. Exit
    """)
    
    while True:
        choice = input("Enter your choice (1-4): ")
        
        if choice == "1":
            app.run_live_mode()
        elif choice == "2":
            filename = input("Enter audio file path (must be WAV format): ")
            app.run_file_mode(filename)
        elif choice == "3":
            app.test_microphone()
        elif choice == "4":
            print("Exiting program...")
            break
        else:
            print("Invalid choice. Please try again.")

if __name__ == "__main__":
    main()


    Speech Recognition Project
    -------------------------
    1. Live microphone mode
    2. Audio file mode
    3. Test microphone
    4. Exit
    


Enter your choice (1-4):  1


System: Speech recognition system activated. How can I help you?
Listening...
Recognizing...
You said: what is date today
System: Today's date is May 03, 2025
Listening...
Recognizing...
System: Sorry, I didn't catch that. Could you repeat?
Listening...
Recognizing...
System: Sorry, I didn't catch that. Could you repeat?
Listening...
Recognizing...
You said: what is time today time what is time today
System: The current time is 11:02
Listening...
Recognizing...
You said: what is artificial intelligence
System: I didn't understand that command. Try saying hello, time, date, or exit.
Listening...
Recognizing...
You said: hello
System: Hello there! How can I help you?
Listening...
Recognizing...
System: Sorry, I didn't catch that. Could you repeat?
Listening...
Recognizing...
You said: time today what is time today
System: The current time is 11:02
Listening...
Recognizing...
You said: what is date today
System: Today's date is May 03, 2025
Listening...
Recognizing...
System: Sorry, I did

Enter your choice (1-4):  2
Enter audio file path (must be WAV format):  artificial intelligence


Error: File artificial intelligence does not exist.


Enter your choice (1-4):  exit


Invalid choice. Please try again.


Enter your choice (1-4):  4


Exiting program...


In [17]:
a=22
a

22

In [19]:
print(a)

22


In [21]:
b=4.5
b

4.5

In [23]:
print(type(b))

<class 'float'>


In [25]:
c=4.5
c

4.5

In [27]:
print(c)

4.5


In [ ]:
prin